# Chapter 10.1–10.3: Initialization and Gradient Flow

This is a recorded lesson for Tensaur (`bedrock`). It runs top to bottom with a fixed seed. You do not need to edit any cell.

**What this chapter asks.** Hidden units that start identical stay identical. Weights that start too large push `tanh` into its flat tails, so gradients vanish. Xavier (and He, for ReLU) pick a random scale that keeps a neuron's weighted sum around size 1.

**The experiment.** We train the same XOR MLP three times and change only how the *hidden* weights start:

1. All zeros
2. Normal with standard deviation 10
3. Xavier, with standard deviation $\sqrt{1/n_{\mathrm{in}}}$

Everything else is held fixed: seed, architecture, learning rate, optimizer, and step count.

**The setup** matches `packages/bedrock/examples/example_tensaur_xor.py`:

- Hidden layer: `Linear(2, 8)` followed by `tanh`
- Output layer: `Linear(8, 1)` followed by `sigmoid`
- Loss: `bce_mean` (binary cross-entropy)
- Optimizer: `Adam` with `lr=0.01` (Adam is in `bedrock.optim` now)
- 1,000 steps, seed `0`

`Linear` has no separate init method, so we overwrite `hidden.weight.data` after construction. Biases already start at 0. Output weights stay at `Linear`'s default (`rng.normal(scale=0.5)`), so only the hidden-weight init changes.

For each run we record:

- Loss at every step
- The fraction of hidden activations with $|\tanh| > 0.99$ on the *first* forward pass
- The hidden-weight gradient norm at step 1, `np.linalg.norm(grad)`
- Whether the hidden units are still identical at the end (max pairwise absolute difference of hidden-weight columns)


## Concept recap

**Zero init and symmetry.** If every hidden unit starts with the same weights (all zeros), every unit computes the same number. Backprop then sends each unit the same update, so they stay clones forever. Your 8 hidden units act like 1.

It can be even worse than identical updates. The error sent back to a hidden unit is *(that unit's output weight)* × *(the output error)*. If the output weights are also 0, that product is 0, the hidden weights get a *zero* gradient, and they never move. Only the output layer could learn — and with identical hidden units it cannot solve XOR.

> **Note.** In this notebook we zero only the hidden weights. The output weights stay at `Linear`'s default random values. The exact all-zero behavior depends on whether those output weights are zero too, and on whether the data's errors cancel. We will look at what XOR actually does rather than assume the textbook special case.

**Large weights and saturation.** `tanh` squishes its input into $(-1, 1)$. Far from 0 it goes flat: the slope is $1 - \tanh^2(z)$. At an output of $0.99$, that slope is $1 - 0.9801 \approx 0.02$. Backprop multiplies by this slope at every `tanh`, so the signal can shrink about $50\times$ per layer. The weights barely move. That is **vanishing gradients**. The opposite failure, **exploding gradients**, is the signal growing layer after layer — it shows up when weights are big and the activation does *not* squash, like ReLU or a linear layer.

**Variance of a sum, then Xavier and He.** Variance measures how spread out a number is.

1. *Scaling.* Multiply a random number by $w$ and its variance grows by $w^2$.
2. *Adding.* Independent random numbers add their variances. $n$ numbers of variance 1 sum to something of variance $n$, so a typical size of about $\pm\sqrt{n}$, not $\pm n$, because some cancel.

A neuron computes $w_1 x_1 + \cdots + w_n x_n$. If each input has variance about 1 and each weight has variance $\sigma^2$, the sum has variance $n\sigma^2$. More inputs give a bigger sum. With $n = 256$ and $\sigma = 1$, the sum is about $\pm 16$, which saturates `tanh` completely.

To keep that variance near 1 you want $n\sigma^2 = 1$, so $\sigma^2 = 1/n_{\mathrm{in}}$. That is the fan-in form of **Xavier initialization** ([Glorot & Bengio, 2010](https://proceedings.mlr.press/v9/glorot10a.html)). (The paper also discusses $2/(n_{\mathrm{in}}+n_{\mathrm{out}})$; both come from the same idea.)

**He initialization** ([He et al., 2015](https://arxiv.org/abs/1502.01852)) is the ReLU version. ReLU sets every negative input to 0, which throws away about half the signal, so you double the variance: $\sigma^2 = 2/n_{\mathrm{in}}$. Use Xavier ($1/n$) for `tanh` or sigmoid and He ($2/n$) for ReLU.

The formula gives *variance*. `randn` multiplies by the standard deviation $\sigma$, so take the square root: Xavier uses $\sigma = \sqrt{1/n_{\mathrm{in}}}$. For this hidden layer $n_{\mathrm{in}} = 2$, so $\sigma = \sqrt{1/2} \approx 0.707$.


## Shared setup

XOR data, the model builder, the three inits, and one training function used by every run below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from bedrock.autograd.tensaur import Tensaur
from bedrock.loss.BCE import bce_mean
from bedrock.nn import Linear
from bedrock.optim import Adam

SEED = 0
N_HIDDEN = 8
LR = 0.01
STEPS = 1000

X = Tensaur(
    [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]],
    requires_grad=False,
)
y = Tensaur([[0.0], [1.0], [1.0], [0.0]], requires_grad=False)

print("XOR inputs:\n", X.data)
print("XOR labels:\n", y.data)
print(f"Hidden size {N_HIDDEN}, Adam lr={LR}, steps={STEPS}, seed={SEED}")
print("ln(2) (BCE of always predicting 0.5) =", float(np.log(2)))

In [ ]:
def build_model() -> tuple[Linear, Linear]:
    """Same 2-8-1 MLP as packages/bedrock/examples/example_tensaur_xor.py."""
    rng = np.random.default_rng(SEED)
    hidden = Linear(2, N_HIDDEN, rng)
    output = Linear(N_HIDDEN, 1, rng)
    return hidden, output


def init_hidden_weights(hidden: Linear, method: str) -> None:
    """Overwrite only the hidden weights. Biases stay 0; the output layer is unchanged.

    The large and Xavier draws share SEED + 1, so they use the same random
    directions and differ only in scale.
    """
    rng = np.random.default_rng(SEED + 1)
    shape = hidden.weight.data.shape
    n_in = shape[0]
    if method == "zeros":
        hidden.weight.data[...] = 0.0
    elif method == "large":
        hidden.weight.data[...] = rng.normal(scale=10.0, size=shape)
    elif method == "xavier":
        hidden.weight.data[...] = rng.normal(scale=np.sqrt(1.0 / n_in), size=shape)
    else:
        raise ValueError(f"Unknown init method: {method}")


def max_pairwise_column_diff(weight: np.ndarray) -> float:
    """Largest absolute difference between any two hidden-unit columns."""
    # weight has shape (n_in, n_out); each column is one hidden unit.
    return float(np.max(np.abs(weight[:, :, None] - weight[:, None, :])))


def train(method: str) -> dict:
    hidden, output = build_model()
    init_hidden_weights(hidden, method)
    optimizer = Adam(hidden.parameters() + output.parameters(), lr=LR)

    losses: list[float] = []
    first_hidden_act = None
    saturation_fraction = None
    step1_hidden_weight_grad = None
    step1_grad_norm = None
    step1_other_grad_norms = None

    for step in range(STEPS):
        optimizer.zero_grad()
        hidden_act = hidden(X).tanh()
        probs = output(hidden_act).sigmoid()
        loss = bce_mean(probs, y)

        if step == 0:
            first_hidden_act = np.array(hidden_act.data, copy=True)
            saturation_fraction = float(np.mean(np.abs(first_hidden_act) > 0.99))

        loss.backward()

        if step == 0:
            step1_hidden_weight_grad = np.array(hidden.weight.grad, copy=True)
            step1_grad_norm = float(np.linalg.norm(step1_hidden_weight_grad))
            step1_other_grad_norms = {
                "hidden.bias": float(np.linalg.norm(hidden.bias.grad)),
                "output.weight": float(np.linalg.norm(output.weight.grad)),
                "output.bias": float(np.linalg.norm(output.bias.grad)),
            }

        optimizer.step()
        losses.append(float(np.asarray(loss.data)))

    final_hidden = hidden(X).tanh()
    final_probs = np.asarray(output(final_hidden).sigmoid().data).reshape(-1)
    max_column_diff = max_pairwise_column_diff(hidden.weight.data)

    return {
        "method": method,
        "losses": np.asarray(losses),
        "final_loss": losses[-1],
        "first_hidden_act": first_hidden_act,
        "saturation_fraction": saturation_fraction,
        "step1_hidden_weight_grad": step1_hidden_weight_grad,
        "step1_grad_norm": step1_grad_norm,
        "step1_other_grad_norms": step1_other_grad_norms,
        "final_probs": final_probs,
        "max_column_diff": max_column_diff,
        "units_identical": max_column_diff < 1e-12,
    }


def print_run(result: dict) -> None:
    print(f"Init: {result['method']}")
    print("First-forward hidden activations (rows are XOR inputs [00, 01, 10, 11]):")
    print(np.round(result["first_hidden_act"], 4))
    print(f"Saturation fraction (|tanh| > 0.99): {result['saturation_fraction']:.6f}")
    print("Step-1 hidden-weight gradient:")
    print(result["step1_hidden_weight_grad"])
    print(f"Step-1 hidden-weight gradient norm: {result['step1_grad_norm']:.6f}")
    other = result["step1_other_grad_norms"]
    print(
        "Step-1 other gradient norms: "
        f"hidden.bias={other['hidden.bias']:.6f}, "
        f"output.weight={other['output.weight']:.6f}, "
        f"output.bias={other['output.bias']:.6f}"
    )
    print(f"Final loss: {result['final_loss']:.6f}")
    print("Final probabilities (targets 0, 1, 1, 0): [" + ", ".join(f"{p:.6f}" for p in result["final_probs"]) + "]")
    print(
        f"Hidden units identical after training? {result['units_identical']} "
        f"(max column |diff| = {result['max_column_diff']:.6f})"
    )


runs = {}
print("Helpers ready.")

## Run A: all zeros

Hidden weights are set to 0. Output weights stay at `Linear`'s default random values.

> **Pause here.** The next cell is already filled in — you do not need to edit anything. Before you look at its output, try to answer:
>
> 1. Will the final loss stay near $\ln 2 \approx 0.693$, or drop?
> 2. Will the saturation fraction be high, low, or zero?
> 3. Will the step-1 hidden-weight gradient norm be zero, tiny, or healthy?
> 4. Will the hidden units still be identical after 1,000 steps?
>
> _Optional scratch space:_  
> _loss: … &nbsp;&nbsp; saturation: … &nbsp;&nbsp; grad norm: … &nbsp;&nbsp; identical: …_


In [ ]:
runs["zeros"] = train("zeros")
print_run(runs["zeros"])

### What actually happened (zeros)

The network did not learn at all. Final loss stayed at **0.693147**, which is exactly $\ln 2$ — the BCE of always predicting 0.5. Every hidden activation on the first forward pass was 0 (`tanh(0) = 0`), so nothing was saturated, and every hidden unit stayed a clone of every other (all weights still 0). The four final probabilities are still `[0.5, 0.5, 0.5, 0.5]`.

The step-1 hidden-weight gradient norm was **exactly 0**. The other parameter gradients were 0 too: hidden bias, output weights, and output bias. Nothing moved for 1,000 steps.

Why, if we only zeroed the hidden weights and left the output weights random? Two things stacked:

1. Hidden activations are all 0, so the output layer sees a blank input. Its weight gradient is *(hidden activations)* × *(output error)* = 0.
2. XOR's four labels are balanced (two 0s, two 1s) and every prediction starts at 0.5. The four output errors are $+e, -e, -e, +e$ and cancel. The output-bias gradient is their sum, which is 0. Each input feature is on for one positive example and one negative example, so those errors cancel in the hidden-weight gradient as well.

The recap said the hidden gradient is exactly 0 when the output weights are also 0. That is the general rule, because the error sent backward is *(output weight)* × *(output error)*. Here the output weights were *not* 0 — they were `Linear`'s default random values — and the hidden gradient was still 0, because XOR's own symmetry cancelled the errors. On a different dataset those random output weights could have broken the deadlock. The reliable way to break symmetry is still small random hidden weights.


## Run B: Normal with std 10

Same model and seed. Hidden weights are drawn from $\mathcal{N}(0, 10^2)$.

> **Pause here.** You do not need to edit the next cell. Before you look at its output, try to answer:
>
> 1. Will many hidden activations saturate ($|\tanh| > 0.99$) on the first forward pass?
> 2. Will the step-1 hidden-weight gradient be tiny (vanishing) or still usable?
> 3. Will the final loss stay near 0.693, or can this tiny XOR net still learn?
> 4. Will the hidden units stay identical?
>
> _Optional scratch space:_  
> _loss: … &nbsp;&nbsp; saturation: … &nbsp;&nbsp; grad norm: … &nbsp;&nbsp; identical: …_


In [ ]:
runs["large"] = train("large")
print_run(runs["large"])

### What actually happened (std 10)

**Saturation showed up, as predicted.** The fraction of hidden activations with $|\tanh| > 0.99$ was **0.656250**, which is $21$ of the $32$ values (4 XOR inputs × 8 hidden units). Look at the activation table: the `[0, 0]` row is all zeros (bias starts at 0, so that input cannot saturate), but most other entries are piled up near $+1$ or $-1$. Big weights pushed `tanh` into its flat tails.

The step-1 hidden-weight gradient norm was **0.024917**, almost the same size as Xavier's **0.027637**. That sounds healthy until you look at the gradient *matrix*: most entries are tiny ($10^{-9}$ to $10^{-6}$) because `tanh`'s slope $1 - \tanh^2$ is about $0.02$ at $0.99$ and nearly $0$ at $0.999999$. A few larger entries (around $0.01$–$0.02$) come from the units that did *not* fully saturate. The overall L2 norm is dominated by those lucky paths, so it hides how many units are nearly stuck.

**The surprise: it still solved XOR.** Final loss was **0.008523**, and the four predictions `[0.000142, 0.996558, 0.986409, 0.016618]` correctly classify all four points. Large init is not automatically fatal on this problem. XOR is only four points with two inputs. The `[0, 0]` example never saturates at step 0, a couple of units randomly land in a less-saturated region, and Adam can follow those paths for 1,000 steps. In a deeper net, or with more inputs (so every sum is a bigger pile of large terms), this would be a much worse idea.

The hidden units did not stay identical — they started different, and the max column difference grew to about **25.15**.


## Run C: Xavier

Hidden weights are drawn from $\mathcal{N}(0, 1/n_{\mathrm{in}})$, so $\sigma = \sqrt{1/2} \approx 0.707$. Same random directions as the std-10 run; only the scale changes.

> **Pause here.** You do not need to edit the next cell. Before you look at its output, try to answer:
>
> 1. Will the saturation fraction be near zero?
> 2. Will the step-1 gradient be spread across many hidden weights, rather than a few survivors?
> 3. Will the final loss beat the std-10 run?
> 4. Will the hidden units stay identical?
>
> _Optional scratch space:_  
> _loss: … &nbsp;&nbsp; saturation: … &nbsp;&nbsp; grad norm: … &nbsp;&nbsp; identical: …_


In [ ]:
runs["xavier"] = train("xavier")
print_run(runs["xavier"])

### What actually happened (Xavier)

No activations saturated (fraction **0.000000**). The first-forward `tanh` values sit in a comfortable range, roughly $\pm 0.7$, which is where `tanh`'s slope is still useful. The step-1 hidden-weight gradient is spread across many entries instead of concentrated in a few survivors, and its norm (**0.027637**) is in the same ballpark as the large-init run without those vanishing entries.

Final loss was **0.000716** — about $12\times$ lower than large init — and the predictions `[0.000895, 0.999113, 0.999490, 0.000564]` are a clean solve. The hidden units specialized (max column difference about **4.98**); they are not clones.

Large and Xavier used the same random directions (same seed) and differed only in scale. Scaling that pattern down from $10$ to $\sqrt{1/2}$ is what kept `tanh` out of the flat tails.


## Comparison

Final metrics for all three runs, then the loss curves on a log scale.


In [ ]:
headers = [
    "init",
    "final loss",
    "saturation",
    "step-1 grad norm",
    "units identical",
    "max column |diff|",
]
rows = []
for key, name in [("zeros", "zeros"), ("large", "std=10"), ("xavier", "Xavier")]:
    r = runs[key]
    rows.append(
        [
            name,
            f"{r['final_loss']:.6f}",
            f"{r['saturation_fraction']:.6f}",
            f"{r['step1_grad_norm']:.6f}",
            str(r["units_identical"]),
            f"{r['max_column_diff']:.6f}",
        ]
    )

widths = [max(len(headers[i]), max(len(row[i]) for row in rows)) for i in range(len(headers))]


def fmt(row):
    return "  ".join(val.ljust(widths[i]) for i, val in enumerate(row))


print(fmt(headers))
print("  ".join("-" * w for w in widths))
for row in rows:
    print(fmt(row))

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = {
    "zeros": "All zeros",
    "large": "Normal (std = 10)",
    "xavier": r"Xavier ($\sigma=\sqrt{1/n_{in}}$)",
}
for key, label in labels.items():
    ax.plot(runs[key]["losses"], label=label)
ax.set_yscale("log")
ax.set_xlabel("Step")
ax.set_ylabel("BCE loss")
ax.set_title("XOR training loss by hidden-weight initialization")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

## Takeaways

- **Zeros keep hidden units as clones, and on this XOR run every gradient was exactly 0.** The network sat at $\ln 2$ for all 1,000 steps. XOR's balanced errors were enough to cancel updates even though the output weights were *not* zero. Small random hidden weights are what break that symmetry.
- **Std 10 saturated about two-thirds of the hidden activations and shrank most per-weight gradients**, but this tiny 2-input net still solved XOR (final loss about 0.0085). The overall gradient *norm* looked similar to Xavier's because a few unsaturated paths dominated the L2 sum. That luck should not be counted on in a deeper network.
- **Xavier kept activations in range, gradients spread out, and reached a final loss about $12\times$ lower** (about 0.00072). Same random directions as std 10, smaller scale. For ReLU, use He ($2/n_{\mathrm{in}}$) instead.

**Interview phrasing.** If every hidden unit starts with the same weights, they compute the same feature and get the same update, so they stay clones — and if the outgoing weights are zero, or the errors cancel the way they do on a balanced problem like XOR, the hidden gradient can be exactly zero and nothing learns. Very large weights push `tanh` into its flat tails, where the local slope $1 - \tanh^2$ is tiny, so most of the gradient vanishes; that is less deadly on a tiny problem like XOR, but it gets worse as nets get deeper or wider. Xavier (and He, for ReLU) pick a random scale whose variance is $1/n_{\mathrm{in}}$ (or $2/n_{\mathrm{in}}$) so a neuron's weighted sum stays order-1, which keeps activations and gradients in a usable range.

**Next: 10.4–10.6** (sigmoid/`tanh` saturation, ReLU backward, diagnosing vanishing, exploding, and nonfinite gradients).
